In [7]:
import sys,os
from openai import OpenAI
import psycopg2
import pandas as pd
from typing import List, Dict

In [2]:
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
POSTGRES_PASS = os.getenv("POSTGRES_PASS")

In [8]:
client = OpenAI(
    api_key=OPENAI_API_KEY
)

In [4]:
def connect_to_db():
    conn = psycopg2.connect(
        dbname="puppet_db",
        user="postgres",
        password=POSTGRES_PASS,
        host="localhost",
        port="5432"
    )
    return conn

In [ ]:
def generate_sql_query(prompt: str) -> str:
    # Use OpenAI's chat completions API to generate the SQL query
    completion = client.chat.completions.create(
        model="gpt-4o", 
        store=True,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return completion.choices[0].message.content

In [24]:
if __name__ == "__main__":
    
    input = "What is the total number of customers grouped by source system name?"

    table_info = """
    Relevant Table Information:
    1. customer:
    - customer_key (TEXT, Primary Key): Unique identifier for each customer.
    - first_name (TEXT): Customer's first name.
    - last_name (TEXT): Customer's last name.
    - source_system_name (TEXT): The system from which the customer data originates.
    - dob (DATE): Customer's date of birth.
    - gender (TEXT): Customer's gender.
    - create_timestamp (TIMESTAMP): The timestamp when the customer was created in the system.

    2. address:
    - address_key (TEXT, Primary Key): Unique identifier for each address.
    - full_address (TEXT): The full address, including street, city, etc.
    - state (TEXT): The state or province of the address.
    - country (TEXT): The country of the address.
    - latitude (TEXT): The latitude of the address.
    - longitude (TEXT): The longitude of the address.

    3. customer_address:
    - customer_key (TEXT, Foreign Key): Refers to the unique identifier of a customer in the 'customer' table.
    - address_key (TEXT, Foreign Key): Refers to the unique identifier of an address in the 'address' table.
    - PRIMARY KEY (customer_key, address_key): Composite primary key.
    """
    top_k = 10

    base_prompt = f"""
            You are a SQL expert with access to a BigQuery dataset containing customers and customer addresses.
            Given an input question, generate a syntactically correct SQL query to answer it. Unless explicitly requested otherwise, limit the results to {top_k} rows.

            Relevant Table Information:
            {table_info}

            Question: {input}

            Guidelines:
            1. Ensure that all attribute searches are case-insensitive.
            2. ALWAYS add 'LIMIT {top_k}' at the end of the query unless:
              - The question explicitly asks for all records
              - The query uses GROUP BY and needs to show all groups
              - The query is counting records (using COUNT)
              - The query calculates aggregates that need all data

            Address and Location Queries:
            1. For questions about addresses, locations, or properties, always include latitude and longitude columns in the SELECT clause.

            Double check the user's postgresql query for common mistakes, including:
            - Using NOT IN with NULL values
            - Using UNION when UNION ALL should have been used
            - Using BETWEEN for exclusive ranges
            - Data type mismatch in predicates
            - Properly quoting identifiers
            - Using the correct number of arguments for functions
            - Casting to the correct data type
            - Using the proper columns for joins
            - Missing LIMIT clause when returning raw records

            If there are any of the above mistakes, rewrite the query.
            If there are no mistakes, just reproduce the original query with no further commentary.

            Provide only the final SQL query as plain text without any formatting.
            If the question is not about customers or addresses, respond with "I don't know"
            """
    
    # print("Prompt:", base_prompt)
    sql_non_parsed = generate_sql_query(base_prompt)
    print(sql_non_parsed)

```sql
SELECT source_system_name, COUNT(*) AS total_customers
FROM customer
GROUP BY source_system_name;
```


In [25]:
sql_non_parsed

'```sql\nSELECT source_system_name, COUNT(*) AS total_customers\nFROM customer\nGROUP BY source_system_name;\n```'

In [28]:
sql_query = sql_non_parsed.strip('```sql\n').strip('\n```')
sql_query

'SELECT source_system_name, COUNT(*) AS total_customers\nFROM customer\nGROUP BY source_system_name;'

In [29]:
conn = psycopg2.connect(
    dbname="puppet_db",
    user="postgres",
    password=POSTGRES_PASS,
    host="localhost",
    port="5432"
)

In [31]:
df = pd.read_sql_query(sql_query, conn)
df

C:\Users\quinb\AppData\Local\Temp\ipykernel_30948\952342913.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql_query, conn)


,source_system_name,total_customers
0,System_C,1
1,System_B,1
2,System_A,2


# Finding best charts

In [ ]:
"""You are an AI assistant that recommends appropriate data visualizations for customer and address analytics. Based on the user's question, SQL query, and query results, suggest the most suitable type of graph or chart to visualize the data.

  Available chart types and their best use cases:

  - Bar Graphs (for 3+ categories): 
    * Comparing distributions across multiple categories
    * Customer counts by source system
    * Customer demographics across regions/states
    * Age group distributions
    * Monthly/yearly registration counts

  - Horizontal Bar Graphs (for 2-3 categories or large value disparities):
    * Binary comparisons (e.g., gender distribution)
    * Limited category comparisons (2-3 items)
    * Cases with large value differences between categories

  - Line Graphs (for time series only):
    * Customer registration trends over time
    * Growth patterns by source system
    * Any metric tracked over time periods
    Note: X-axis MUST represent time (create_timestamp or similar)

  - Pie Charts (for proportions, 3-7 categories max):
    * Distribution percentages
    * Market share analysis
    * Proportional comparisons
    Note: Total should sum to 100%

  - Scatter Plots (for numeric relationships):
    * Age vs other numeric metrics
    * Timestamp patterns
    * Distribution analysis
    Note: Both axes must be numeric, non-categorical

  Special Cases:
  1. Geographic Data:
    * If result contains latitude and longitude → No chart (will display map)
    * For address/location questions → No chart (will display map)

  2. Raw Data:
    * Individual customer records → No chart (tabular display)
    * Non-aggregated data → No chart (tabular display)

  Tables in scope:
  - customer: customer_key, first_name, last_name, source_system_name, dob, gender, create_timestamp
  - customer_address: customer_key, address_key
  - address: address_key, full_address, state, country, latitude, longitude

  Question: {question}
  SQL Query: {query}
  SQL Result: {result}

  Provide your response in the following format:
  Recommended Visualization: [Chart type or "None"]. ONLY use the following names: bar, horizontal_bar, line, pie, scatter, none
  Reason: [Brief explanation for your recommendation]
  """

In [ ]:
"""Given the following user question, corresponding SQL query, and SQL result, answer the user question.

    Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer: """